In [ ]:
%pip install beautifulsoup4 requests polars

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

If you really know what your doing, you can silence this warning with the warning module
or by setting POLARS_ALLOW_FORKING_THREAD=1.

  pid, fd = os.forkpty()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# SP500_INCEPTION is not defined, define it before use
import json


SP500_INCEPTION = '1957-03-04'

# Storage paths for landing zone
SECURITIES_PATH = "data/landing/securities"
PRICES_PATH = "data/landing/prices"

def get_sp500_tickers():
    print("Fetching S&P 500 constituents from Wikipedia...")
    import polars as pl
    import requests
    from bs4 import BeautifulSoup
    from datetime import datetime

    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'})
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', {'id': 'constituents'})
    rows = table.find_all('tr')

    columns = [th.text.strip() for th in rows[0].find_all('th')]
    data = []
    for row in rows[1:]:
        cells = [td.text.strip() for td in row.find_all('td')]
        
        if len(cells) == len(columns):
            data.append(cells)

    df = pl.DataFrame(data, schema={col: pl.Utf8 if col not in ('date_added', 'updated_at') else pl.Datetime for col in columns})

    securities = []
    for row in df.iter_rows(named=True):
        security = {
            'ticker': row['Symbol'].replace('.', '-'),
            'name': row['Security'],
            'status': 'ACTIVE',
            'sector': row['GICS Sector'],
            'subsector': row['GICS Sub-Industry'],
            'date_added': str(row['Date added']) if row['Date added'] else None,
            'updated_at': datetime.now().isoformat()
        }
        securities.append(security)
    
    return securities
    
def fetch_price_data(ticker, start_date=SP500_INCEPTION):
    try:
        import yfinance as yf
        import polars as pl
        from datetime import datetime

        stock = yf.Ticker(ticker)
        hist = stock.history(start=start_date, end=datetime.now().strftime('%Y-%m-%d'))
        
        if hist.empty:
            return []
        
        df = pl.DataFrame(hist.reset_index())
        prices = []
        for row in df.iter_rows(named=True):
            price = {
                'ticker': ticker,
                'date': row['Date'].strftime('%Y-%m-%d') if hasattr(row['Date'], 'strftime') else str(row['Date']),
                'open': float(row['Open']) if row['Open'] is not None else None,
                'high': float(row['High']) if row['High'] is not None else None,
                'low': float(row['Low']) if row['Low'] is not None else None,
                'close': float(row['Close']) if row['Close'] is not None else None,
                'adj_close': float(row.get('Adj Close', row['Close'])) if row.get('Adj Close', row['Close']) is not None else None,
                'volume': int(row['Volume']) if row['Volume'] is not None and row['Volume'] > 0 else None,
                'dividends': float(row.get('Dividends', 0)) if row.get('Dividends', 0) is not None else None,
                'stock_splits': float(row.get('Stock Splits', 0)) if row.get('Stock Splits', 0) is not None else None,
                'updated_at': datetime.now().isoformat()
            }
            prices.append(price)
        
        return prices
    except Exception as e:
        print(f"Error fetching {ticker}: {str(e)}")
        return []

def write_to_landing_zone(securities, prices_dict):
    import json
    from datetime import datetime

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    json.dump(securities, open(f"{SECURITIES_PATH}/securities_{timestamp}.json", 'w'), indent=2)        
    print(f"✓ Written {len(securities)} securities")
    
    total_prices = 0
    for ticker, prices in prices_dict.items():
        if prices:
            prices_content = '\n'.join([json.dumps(p) for p in prices])
            with open(f"{PRICES_PATH}/prices_{ticker}_{timestamp}.jsonl", 'w') as f:
                f.write(prices_content)
            total_prices += len(prices)
    
    print(f"✓ Written {total_prices:,} price records")

print("="*60)
print("S&P 500 Data Fetcher")
print("="*60)

securities = get_sp500_tickers()
print(f"\n✓ Found {len(securities)} securities")

print("\nFetching price data (10-15 minutes)...")
prices_dict = {}

import time

for i, security in enumerate(securities):
    ticker = security['ticker']
    print(f"[{i+1}/{len(securities)}] {ticker}...", end=' ')
    
    prices = fetch_price_data(ticker)
    if prices:
        prices_dict[ticker] = prices
        print(f"✓ {len(prices)}")
    else:
        print("✗")
    
    if (i + 1) % 10 == 0:
        time.sleep(1)

write_to_landing_zone(securities, prices_dict)
print("\n✓ Complete! Pipeline ready to run.")

S&P 500 Data Fetcher
Fetching S&P 500 constituents from Wikipedia...


/var/folders/tb/2z_lc5nd1cn_xhn5dcw7t6jr0000gp/T/ipykernel_90190/2818208134.py:32: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(data, schema={col: pl.Utf8 if col not in ('date_added', 'updated_at') else pl.Datetime for col in columns})



✓ Found 503 securities

Fetching price data (10-15 minutes)...
[1/503] MMM... Error fetching MMM: No module named 'yfinance'
✗
[2/503] AOS... Error fetching AOS: No module named 'yfinance'
✗
[3/503] ABT... Error fetching ABT: No module named 'yfinance'
✗
[4/503] ABBV... Error fetching ABBV: No module named 'yfinance'
✗
[5/503] ACN... Error fetching ACN: No module named 'yfinance'
✗
[6/503] ADBE... Error fetching ADBE: No module named 'yfinance'
✗
[7/503] AMD... Error fetching AMD: No module named 'yfinance'
✗
[8/503] AES... Error fetching AES: No module named 'yfinance'
✗
[9/503] AFL... Error fetching AFL: No module named 'yfinance'
✗
[10/503] A... Error fetching A: No module named 'yfinance'
✗
[11/503] APD... Error fetching APD: No module named 'yfinance'
✗
[12/503] ABNB... Error fetching ABNB: No module named 'yfinance'
✗
[13/503] AKAM... Error fetching AKAM: No module named 'yfinance'
✗
[14/503] ALB... Error fetching ALB: No module named 'yfinance'
✗
[15/503] ARE... Error fetching 